# Group 12 — PLSA Information Retrieval Project (All 20 Categories)
This notebook trains PLSA on **all 20 categories** of 20 Newsgroups. It implements PLSA with NumPy/EM, inspects topics, tests controlled mixtures from two selected categories, performs query folding-in, and compares retrieval against TF-IDF/VSM and LSI.

**Main experiment:** all 20 categories. **Boundary experiment:** a controlled mixture of two source categories (baseball and politics), as specified in the proposal. Category labels are not used to train PLSA; they are used for evaluation and selecting synthetic source documents.


## 0. Install dependencies

Run once in a notebook environment if required. In Colab, use the first line in a code cell.

In [ ]:
# %pip -q install numpy scipy scikit-learn pandas matplotlib

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)


## 2. Load the selected dataset

We load only the two categories selected for the topic-boundary experiment. The category labels are used for evaluation and for choosing source documents for synthetic mixtures; PLSA itself is unsupervised and does not receive these labels during training.

In [ ]:
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=SEED
)
documents = np.array(dataset.data, dtype=object)
labels = np.asarray(dataset.target)
target_names = list(dataset.target_names)

print("Number of categories:", len(target_names))
print("Total documents:", len(documents))
for i, name in enumerate(target_names):
    print(f"{i:2d} | {name:25s} | {(labels == i).sum():5d} documents")

### Main corpus versus boundary experiment

The model and retrieval evaluation use the **full 20-category corpus**. The controlled boundary experiment still uses two chosen source categories to create interpretable 90:10 through 10:90 word mixtures. This does not mean the PLSA model was trained on only those two categories.

## 3. Train/test split and preprocessing

Split the raw documents first. Fit the vocabulary only on the training split to avoid vocabulary leakage from the test documents.

In [ ]:
import re
import spacy
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

docs_train, docs_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.25,
    random_state=SEED,
    stratify=labels
)
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]
)
def preprocess_texts(texts):
    """
    Lowercase, tokenize, remove stop words and punctuation,
    and lemmatize the remaining tokens.

    Returns a list of preprocessed document strings.
    """
    processed_docs = []

    for doc in nlp.pipe(texts, batch_size=256):
        tokens = []

        for token in doc:

            if (
                token.is_space
                or token.is_punct
                or token.like_num
                or not token.is_alpha
            ):
                continue


            if token.is_stop:
                continue


            lemma = token.lemma_.lower().strip()


            if len(lemma) < 2:
                continue

            tokens.append(lemma)

        processed_docs.append(" ".join(tokens))

    return processed_docs
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer = CountVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    max_features=3000,
    min_df=3,
    max_df=0.90,
    dtype=np.float64
)
X_train = count_vectorizer.fit_transform(
    docs_train_processed
)
X_test = count_vectorizer.transform(
    docs_test_processed
)
vocabulary = count_vectorizer.get_feature_names_out()
print("X_train:", X_train.shape, "(documents, vocabulary)")
print("X_test: ", X_test.shape)
print("Example vocabulary:", vocabulary[:20])

print(
    "Empty training documents after preprocessing:",
    sum(not doc.strip() for doc in docs_train_processed)
)

print(
    "Empty test documents after preprocessing:",
    sum(not doc.strip() for doc in docs_test_processed)
)


## 4. PLSA theory

For document \(d\), word \(w\), and latent topic \(z\):

\[
P(w\mid d)=\sum_{z=1}^{K}P(w\mid z)P(z\mid d)
\]

- \(\theta_{d,z}=P(z\mid d)\): document-topic distribution.
- \(\phi_{z,w}=P(w\mid z)\): topic-word distribution.
- Each row of \(\theta\) and \(\phi\) sums to one.

For every observed document-word pair, the E-step computes the posterior responsibility:

\[
P(z\mid d,w)=\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'}P(z'\mid d)P(w\mid z')}
\]

The M-step uses the expected word counts to update both distributions. The process repeats until the log-likelihood stops improving.

In [ ]:
class PLSA:
    """PLSA trained directly with NumPy EM and sparse input counts."""

    def __init__(self, n_topics=5, max_iter=100, tol=1e-4,
                 random_state=42, beta=1.0):
        if n_topics < 1:
            raise ValueError("n_topics must be positive")
        if not (0 < beta):
            raise ValueError("beta must be positive")
        self.n_topics = n_topics
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.beta = beta
        self.log_likelihoods = []

    @staticmethod
    def _normalize(a, axis):
        denom = np.maximum(a.sum(axis=axis, keepdims=True), 1e-15)
        return a / denom

    @staticmethod
    def _observations(X):
        X = csr_matrix(X, dtype=np.float64)
        X.eliminate_zeros()
        X.sum_duplicates()
        docs, words, counts = [], [], []
        for d in range(X.shape[0]):
            start, end = X.indptr[d], X.indptr[d + 1]
            if end > start:
                docs.append(np.full(end-start, d, dtype=np.int64))
                words.append(X.indices[start:end].astype(np.int64))
                counts.append(X.data[start:end].astype(np.float64))
        if not docs:
            raise ValueError("Input has no observed word counts.")
        return X, np.concatenate(docs), np.concatenate(words), np.concatenate(counts)

    def fit(self, X):
        X, doc_ids, word_ids, counts = self._observations(X)
        D, V = X.shape
        K = self.n_topics
        rng = np.random.default_rng(self.random_state)

        theta = self._normalize(rng.random((D, K)) + 1e-3, axis=1)
        phi = self._normalize(rng.random((K, V)) + 1e-3, axis=1)
        self.log_likelihoods = []

        for iteration in range(self.max_iter):

            unnorm = theta[doc_ids].T * phi[:, word_ids]
            if self.beta != 1.0:
                unnorm = np.maximum(unnorm, 1e-15) ** self.beta
            posterior = unnorm / np.maximum(
                unnorm.sum(axis=0, keepdims=True), 1e-15
            )


            weighted = posterior * counts[None, :]
            topic_word = np.zeros((K, V), dtype=np.float64)
            doc_topic = np.zeros((D, K), dtype=np.float64)

            for k in range(K):
                np.add.at(topic_word[k], word_ids, weighted[k])
                np.add.at(doc_topic[:, k], doc_ids, weighted[k])

            phi = self._normalize(topic_word + 1e-12, axis=1)
            theta = self._normalize(doc_topic + 1e-12, axis=1)


            p_observed = np.sum(theta[doc_ids] * phi[:, word_ids].T, axis=1)
            ll = float(np.sum(counts * np.log(np.maximum(p_observed, 1e-15))))
            self.log_likelihoods.append(ll)

            if iteration > 0:
                improvement = ll - previous_ll
                if self.beta == 1.0 and improvement < -1e-7:
                    print("Warning: likelihood decreased slightly; check numerical behavior.")
                if abs(improvement) <= self.tol * (1 + abs(previous_ll)):
                    break
            previous_ll = ll

        self.theta_ = theta
        self.phi_ = phi
        self.n_features_in_ = V
        self.n_iter_ = len(self.log_likelihoods)
        return self

    def transform(self):
        """Document-topic probabilities for the documents used in fit()."""
        return self.theta_

    def fold_in(self, X_new, max_iter=100):
        """Infer P(z|d_new) with learned P(w|z) fixed."""
        X_new = csr_matrix(X_new, dtype=np.float64)
        if X_new.shape[1] != self.n_features_in_:
            raise ValueError("New data must use the same fitted vocabulary.")
        D, V = X_new.shape
        K = self.n_topics
        rng = np.random.default_rng(self.random_state + 1000)
        theta = self._normalize(rng.random((D, K)) + 1e-3, axis=1)

        for _ in range(max_iter):
            doc_topic = np.zeros((D, K), dtype=np.float64)
            for d in range(D):
                start, end = X_new.indptr[d], X_new.indptr[d + 1]
                words = X_new.indices[start:end]
                counts = X_new.data[start:end]
                if len(words) == 0:
                    continue
                unnorm = theta[d, :, None] * self.phi_[:, words]
                if self.beta != 1.0:
                    unnorm = np.maximum(unnorm, 1e-15) ** self.beta
                posterior = unnorm / np.maximum(
                    unnorm.sum(axis=0, keepdims=True), 1e-15
                )
                doc_topic[d] = (posterior * counts[None, :]).sum(axis=1)
            theta = self._normalize(doc_topic + 1e-12, axis=1)
        return theta

    def top_words(self, vocabulary, n=10):
        result = {}
        for k, dist in enumerate(self.phi_):
            ids = np.argsort(dist)[::-1][:n]
            result[k] = [(vocabulary[i], float(dist[i])) for i in ids]
        return result

## 5. Multiple random restarts

EM can settle at different local optima. Fit several initializations and select the model with the largest training log-likelihood. This selection criterion is only about training fit; final retrieval quality must still be evaluated on held-out documents.

In [ ]:
def train_plsa_restarts(X, n_topics=5, n_restarts=5,
                        max_iter=100, tol=1e-4, base_seed=42):
    runs = []
    for r in range(n_restarts):
        model = PLSA(
            n_topics=n_topics, max_iter=max_iter, tol=tol,
            random_state=base_seed + r, beta=1.0
        ).fit(X)
        final_ll = model.log_likelihoods[-1]
        runs.append((model, final_ll))
        print(f"Restart {r+1}: final training log-likelihood={final_ll:.3f}, "
              f"iterations={model.n_iter_}")
    best_model, best_ll = max(runs, key=lambda item: item[1])
    print("Selected best training likelihood:", round(best_ll, 3))
    return best_model, runs

K = 5
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)

## 6. Inspect topics and convergence

Topic IDs are arbitrary: topic 0 is not inherently sports or politics. Interpret each topic by its highest-probability words, and use category-level averages only as a diagnostic.

In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()